In [1]:
from pathlib import Path
from test import BOUTHESELInfo1

hesel_path = Path.cwd().resolve().parent / "simulatorer/BOUT/BOUT-HESEL"
data_folder = "data"

info1 = BOUTHESELInfo1(hesel_path, data_folder)

In [2]:
data0 = info1._load_data(0)
data1 = info1._load_data(1)
data2 = info1._load_data(2)
data3 = info1._load_data(3)

In [2]:
for k, v in info1.data.items():
    print(k, v.shape)

lnn (26, 34, 128)
lnpe (26, 34, 128)
lnpi (26, 34, 128)
vort (26, 34, 128)
phi (26, 34, 128)
init_n (34,)
init_pe (34,)
init_pi (34,)
sigma_open (34,)
sigma_closed (34,)
sigma_force (34,)
B (34,)
t_array (26,)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, Dropdown

fields = ["lnn", "lnpe", "lnpi", "vort", "phi"]
chunks = [data0, data1, data2, data3]
overlap = 2
expected_nx = 128


def stitch_x(field_name, chunk_dicts, overlap=2, expected_nx=None):
    """Stack processor chunks along x while removing duplicated overlap cells."""
    arrays = [d[field_name] for d in chunk_dicts]

    # Candidate A: periodic-style trim (drop overlap from every chunk).
    candidate_periodic = np.concatenate([a[:, :-overlap, :] for a in arrays], axis=1)

    # Candidate B: non-periodic-style trim (drop overlap except for last chunk).
    candidate_non_periodic = np.concatenate(
        [a[:, :-overlap, :] for a in arrays[:-1]] + [arrays[-1]], axis=1
    )

    if expected_nx is None:
        return candidate_periodic

    if candidate_periodic.shape[1] == expected_nx:
        return candidate_periodic
    if candidate_non_periodic.shape[1] == expected_nx:
        return candidate_non_periodic

    # Fallback: pick the candidate closest to expected_nx.
    if abs(candidate_periodic.shape[1] - expected_nx) <= abs(candidate_non_periodic.shape[1] - expected_nx):
        return candidate_periodic
    return candidate_non_periodic


stitched = {f: stitch_x(f, chunks, overlap=overlap, expected_nx=expected_nx) for f in fields}
nt, nx, nz = stitched[fields[0]].shape


def plot_xz(t=0, field="lnn"):
    arr = stitched[field][t]  # shape (nx, nz)
    plt.figure(figsize=(8, 4.5))
    plt.imshow(
        arr.T,
        origin="lower",
        aspect="auto",
        extent=[0, nx - 1, 0, nz - 1],
        cmap="viridis",
    )
    plt.xlabel("x")
    plt.ylabel("z")
    plt.title(f"{field} at t={t} (shape={arr.shape})")
    plt.colorbar(label=field)
    plt.tight_layout()
    plt.show()


interact(
    plot_xz,
    t=IntSlider(min=0, max=nt - 1, step=1, value=0, description="t"),
    field=Dropdown(options=fields, value="lnn", description="field"),
);

interactive(children=(IntSlider(value=0, description='t', max=25), Dropdown(description='field', options=('lnn…